# RADICAL-Cybertools: RADICAL-Pilot

One has to handle RADICAL-Pilot applications with some care when running them in a Jupyter notebook.  In particular one should avoid to run cells out of order.  It is usually best to cleanly terminate the kernel before rerunning any / all cells.  This notebook thus puts the exercise code into a *single* cell which you can edit freely and then execute.

## Exercise 1: Change the workload size and composition
  - submit more than one task type
    - change number of tasks
    - change time to sleep
    - change executable (maybe to `/bin/sleep`?)
    - change number or cores per rank
  - use the [RP documentation](https://radicalpilot.readthedocs.io/en/stable/tutorials/describing_tasks.html) for reference


In [1]:
%env RADICAL_REPORT=TRUE
%env RADICAL_REPORT_ANIME=FALSE
%env RADICAL_CONFIG_USER_DIR=/tutorials/local/seavea-hackathon-2025-2/

!radical-stack
!python3 -c 'import radical.pilot as rp; print(rp.__file__)'

import dragon
print(dragon.__file__)

# import flux
# print(flux.__file__)

env: RADICAL_REPORT=TRUE
env: RADICAL_REPORT_ANIME=FALSE
env: RADICAL_CONFIG_USER_DIR=/tutorials/local/seavea-hackathon-2025-2/

  python               : /opt/conda/bin/python3
  pythonpath           : 
  version              : 3.11.10
  virtualenv           : base

  radical.analytics    : 1.103.0-v1.102.0-52-g795f7da@devel
  radical.asyncflow    : Error: module 'radical.asyncflow' has no attribute 'version'
  radical.entk         : 1.103.0-v1.102.0-3-gc1f687bf@devel
  radical.gtod         : 1.103.0-v1.102.0-1-g3dc2d07@devel
  radical.pilot        : 1.103.0-v1.102.0-293-g510d8130e@devel
  radical.utils        : 1.103.0-v1.102.0-75-g5c86afba@devel

  rc.process           : 0.11.0

/opt/conda/lib/python3.11/site-packages/radical/pilot/__init__.py
/opt/conda/lib/python3.11/site-packages/dragon/__init__.py


In [2]:
import radical.pilot as rp
import radical.utils as ru

report = ru.Reporter(name='radical.pilot')
report.title('Getting Started (RP version %s)' % rp.version)

session = rp.Session()
pmgr    = rp.PilotManager(session=session)
tmgr    = rp.TaskManager(session=session)

%env SID={session.uid}


 Getting Started (RP version 1.102.0)                                           

new session: [rp.session.4d3b355a-b950-11f0-bc4b-4e377d3d5e37]                 \
zmq proxy  : [tcp://172.17.0.2:10001]                                         ok
create pilot manager                                                          ok
create task manager                                                           ok


env: SID=rp.session.4d3b355a-b950-11f0-bc4b-4e377d3d5e37


In [3]:
report.header('submit pilot')
pdesc = rp.PilotDescription({'resource'     : 'debug.dragon',
                             'runtime'      : 30,  # pilot runtime minutes
                             'project'      : None,
                             'queue'        : None,
                             'cores'        : 4,
                             'gpus'         : 0,
                             'exit_on_error': False})
pilot = pmgr.submit_pilots(pdesc)
tmgr.add_pilots(pilot)

report.header('execute workload')

n = 10

report.progress_tgt(n, label='create')
tds = list()
for i in range(n):

    td = rp.TaskDescription()
    td.executable     = 'radical-pilot-hello.sh'
    td.arguments      = [10]
    td.ranks          = 1
    td.cores_per_rank = 1

    tds.append(td)
    report.progress()

report.progress_done()

tasks = tmgr.submit_tasks(tds)
tmgr.wait_tasks()

report.header('inspect results')
for task in tasks:
    print('  * %s: %s' % (task.uid, task.state))


--------------------------------------------------------------------------------
submit pilot                                                                    

submit 1 pilot(s)
        pilot.0000   debug.dragon              4 cores       0 gpus           ok

--------------------------------------------------------------------------------
execute workload                                                                

create: ########################################################################
submit: ########################################################################
wait  : ########################################################################
	DONE      :    10
                                                                              ok

--------------------------------------------------------------------------------
inspect results                                                                 



  * task.000000: DONE
  * task.000001: DONE
  * task.000002: DONE
  * task.000003: DONE
  * task.000004: DONE
  * task.000005: DONE
  * task.000006: DONE
  * task.000007: DONE
  * task.000008: DONE
  * task.000009: DONE


In [4]:
report.header('finalize')
session.close()


--------------------------------------------------------------------------------
finalize                                                                        

closing session rp.session.4d3b355a-b950-11f0-bc4b-4e377d3d5e37                \
close task manager                                                            ok
close pilot manager                                                            \
wait for 1 pilot(s)
                                                                              ok
                                                                              ok
session lifetime: 51.9s                                                       ok


In [5]:
!ls $HOME/radical.pilot.sandbox/$SID/pilot*/task.000000/

task.000000.exec.sh  task.000000.files	task.000000.ofiles  task.000000.prof


In [ ]:
!cat `ls $HOME/radical.pilot.sandbox/$SID/pilot*/task.000000/task.000000.prof`